## 데이터준비

In [3]:
#한국 정상 URL 500개 만들기 (코드)
import pandas as pd
import random

# 기본 정상 URL
base_urls = [
    "https://www.naver.com", "https://naver.com",
    "https://www.daum.net", "https://daum.net",
    "https://www.kakao.com", "https://kakao.com",
    "https://www.nate.com", "https://nate.com",
    "https://google.co.kr", "https://www.google.co.kr",

    "https://www.coupang.com", "https://coupang.com",
    "https://www.gmarket.co.kr", "https://www.11st.co.kr",

    "https://www.kbfg.com", "https://obank.kbstar.com",
    "https://www.shinhan.com", "https://bank.shinhan.com",
    "https://www.wooribank.com", "https://www.ibk.co.kr",
    "https://www.nhbank.com", "https://www.citibank.co.kr",

    "https://www.gov.kr", "https://www.hometax.go.kr",
    "https://www.nts.go.kr", "https://www.korea.kr"
]

# 랜덤 path를 붙여 URL을 늘리는 패스 리스트
paths = [
    "/", "/main", "/home", "/login", "/search", "/news",
    "/support", "/help", "/customer", "/about", "/event",
    "/item", "/products"
]

# 500개 만들기
extended = []
for url in base_urls:
    extended.append(url)
    for _ in range(8):  # URL마다 8개 만들어서 80*9 = 720개 → 나중에 500개만 사용
        extended.append(url + random.choice(paths))

extended = extended[:500]

# DataFrame 생성
df_normal_korea = pd.DataFrame({
    "url": extended,
    "label": 0
})

df_normal_korea.head(), df_normal_korea.shape
df_normal_korea.to_csv("normal_korea_500.csv", index=False, encoding="utf-8-sig")



In [2]:
import zipfile, os

zip_path = "phishing URL dataset.zip"  # 다운로드한 파일 이름 맞게 변경
extract_path = "data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ 압축 해제 완료:", os.listdir(extract_path))

✅ 압축 해제 완료: ['Phishing URL dataset']


In [5]:
import os

print(os.listdir("data/Phishing URL dataset"))


['downsampled_dataset.csv', 'Phishing URLs.csv', 'URL dataset.csv']


In [6]:
## 데이터셋 불러와서 형태확인
import pandas as pd

df1 = pd.read_csv(r"C:/Users/monbe/Phising/data/Phishing URL dataset/Phishing URLs.csv")
df2 = pd.read_csv(r"C:/Users/monbe/Phising/data/Phishing URL dataset/URL dataset.csv")

print("df1 컬럼:", df1.columns.tolist())## 피싱
print("df2 컬럼:", df2.columns.tolist())## 정상

df1.head(), df2.head()


df1 컬럼: ['url', 'Type']
df2 컬럼: ['url', 'type']


(                                                 url      Type
 0  https://docs.google.com/presentation/d/e/2PACX...  Phishing
 1    https://btttelecommunniccatiion.weeblysite.com/  Phishing
 2                        https://kq0hgp.webwave.dev/  Phishing
 3  https://brittishtele1bt-69836.getresponsesite....  Phishing
 4         https://bt-internet-105056.weeblysite.com/  Phishing,
                          url        type
 0     https://www.google.com  legitimate
 1    https://www.youtube.com  legitimate
 2   https://www.facebook.com  legitimate
 3      https://www.baidu.com  legitimate
 4  https://www.wikipedia.org  legitimate)

In [7]:
## 두 csv를 하나로 합치기 -> 학습용 통합 데이터셋 만들어짐
import pandas as pd

# 파일 읽기
df_phishing = pd.read_csv(r"C:/Users/monbe/Phising/data/Phishing URL dataset/Phishing URLs.csv")
df_legit = pd.read_csv(r"C:/Users/monbe/Phising/data/Phishing URL dataset/URL dataset.csv")

# 컬럼 이름統一
df_phishing.columns = ['url', 'label']
df_legit.columns = ['url', 'label']

# 라벨 값 변환
df_phishing['label'] = 1        # 피싱 (1)
df_legit['label'] = 0           # 정상 (0)

# 합치기
df = pd.concat([df_phishing, df_legit], axis=0).reset_index(drop=True)

print("총 데이터 수:", len(df))
df.head()


총 데이터 수: 504983


,url,label
0,https://docs.google.com/presentation/d/e/2PACX...,1
1,https://btttelecommunniccatiion.weeblysite.com/,1
2,https://kq0hgp.webwave.dev/,1
3,https://brittishtele1bt-69836.getresponsesite....,1
4,https://bt-internet-105056.weeblysite.com/,1


In [8]:
# 데이터 정제 (중복/결측 제거)
df = df.drop_duplicates(subset=['url']) # 중복행 제거
df = df.dropna(subset=['url', 'label']) # 결측갑 제거
df = df.reset_index(drop=True) # DataFrame의 인덱스 재설정

print("정제 완료:", df.shape)
df.head()

정제 완료: (504933, 2)


,url,label
0,https://docs.google.com/presentation/d/e/2PACX...,1
1,https://btttelecommunniccatiion.weeblysite.com/,1
2,https://kq0hgp.webwave.dev/,1
3,https://brittishtele1bt-69836.getresponsesite....,1
4,https://bt-internet-105056.weeblysite.com/,1


In [9]:
# 기존df에 새로만든 df_normal_korea 합치기
df_v2 = pd.concat([df, df_normal_korea], ignore_index=True)
df_v2 = df_v2.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_v2.shape)
df_v2.head()


(505167, 2)


,url,label
0,https://www.jmusicology.wordpress.com/2011/07/...,0
1,https://www.virginiaimages.com/catalog/norfolk...,0
2,https://www.vervemusicgroup.com/davidbenoit/,0
3,https://www.tv.com/people/eddie-johnston/,0
4,https://www.places.designobserver.com/feature/...,0


In [11]:
# 라벨 분포 확인
df['label'].value_counts()


#----문제 :데이터 불군형이 심함ㅠ (정상이 갯수가 훨씬 많아서 피싱탐지가 안됨) 해결 : 정상데이터 다운샘플링  

0    450128
1     54805
Name: label, dtype: int64

In [8]:
# 해결: 다운샘플링
from sklearn.utils import resample

# 피싱 / 정상 분리
df_phish = df[df['label'] == 1]
df_legit = df[df['label'] == 0]

# 정상 데이터 다운샘플링
df_legit_down = resample(df_legit,
                         replace=False,
                         n_samples=len(df_phish)*1,  # 피싱과 동일 개수로
                         random_state=42)

# 합치기
df_balanced = pd.concat([df_phish, df_legit_down]).sample(frac=1, random_state=42).reset_index(drop=True)

print("샘플링 후 라벨 분포:")
print(df_balanced['label'].value_counts())

df_balanced.to_csv("downsampled_dataset.csv", index=False)


샘플링 후 라벨 분포:
0    54805
1    54805
Name: label, dtype: int64


In [1]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 피싱URL탐지,모델 학습

In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\monbe\Phising\data\Phishing URL dataset\downsampled_dataset.csv"
)


df.head()


,url,label
0,https://www.pittsburghsymphony.org/pghsymph.ns...,0
1,http://onceaunicorn.com/wp-includes/pomo/file/...,0
2,https://www.lifescript.com/doctor-directory/ho...,0
3,https://storage.googleapis.com/trustusaa/signi...,0
4,https://btinternet-104787.weeblysite.com/,1


In [2]:
# URL 텍스트 데이터 -> TF-IDF 벡터화
# URL텍스트를 숫자 벡터로 변환

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5))
# 1. anlayzer='char_wb' : url을 단어가 아니라 한글자로 쪼갬 이유는 url은 단어보다 문자가 중요
# 2. ngram_range=(3,5) : URL을 3~5글자씩 묶어서 분석
# 이런 문자열 패턴을 바탕으로 URL 특징을 벡터화함.

X = vectorizer.fit_transform(df['url'])
# TF-IDF 적용 : 모든 URL을 숫자 벡터로 바꾼 결과가 X

y = df['label']
# 라벨분리

X.shape, y.shape
#X.shape= (샘플수 , TF-IDF차원수) ex(109610,2667914)
# 109610개 URL이 있고 각 URL이 약 2667914 차원의 벡터로 변환
# 즉 한 URL을 2667914의 숫자로 표현했다는 의미야.

#y.shpae = (샘플수,) ex(109610,)
# 109610개의 라벨(정상/피싱)만 있다는것
# X의 109610행과 y의 109610개 라벨이 1:1로 대응된다.

((109610, 2667914), (109610,))

In [4]:
## Train/Test 분리

from sklearn.model_selection import train_test_split
#train_test_split : 데이터를 두세트로 나눔  
# X_train / y_train → 모델 학습용, X_test / y_test → 모델 성능 평가용


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, #전체 데이터의 20%를 테스트(평가)용으로 사용, 나머지 80%는 학습용
    stratify=y, #!! 중요 라벨(0/1) 비율을 똑같이 유지하면서 나눠준다.
    random_state=42 # 데이터를 섞을 때 항상 같은 방식으로 섞기 위함
) 

X_train.shape, X_test.shape

#((샘플수_train, TF-IDF 벡터차원수), (샘플수_test, TF-IDF 벡터차원수))
#ex((6400,500000),(1600,500000))
# X_train.shape = (6400, 500000) 학습 데이터 6400개, 각 URL이 50만 차원짜리 벡터
# X_test.shape = (1600, 500000) 평가 데이터 1600개, 동일하게 50만 차원 벡터
# TF-IDF는 "특징 수(벡터 차원)"가 그대로 유지됨.


((87688, 2667914), (21922, 2667914))

In [5]:
## 모델 선택 (Logistic Regression)
from sklearn.linear_model import LogisticRegression


model = LogisticRegression(max_iter=2000, n_jobs=-1)
#max_iter=2000 : 모델이 수렴할 때까지 최대 2000번 반복해서 학습해라
# → 복잡한 TF-IDF 벡터를 다루기 때문에 반복 횟수를 늘려야 안정됨
#n_jobs = -1 ->  컴퓨터의 모든 CPU 코어를 사용해 계산 더 빠르게 하라

model.fit(X_train, y_train)
#진짜 학습하는 단계 
#X_train: TF-IDF로 숫자로 변환된 URL, y_train: 정상/피싱 라벨을 보고 패턴을 배움.
#“이 URL 문자열 조합은 피싱일 확률이 높구나…이 패턴은 정상일 때 많이 나오네…”
# 이런 식으로 피싱/정상을 구분하는 규칙을 스스로 학습함.
# 이 한 줄에서 진짜 머신러닝이 일어남.

model

LogisticRegression(max_iter=2000, n_jobs=-1)

In [6]:
## 학습 + 모델 평가

from sklearn.metrics import classification_report

pred = model.predict(X_test)
print(classification_report(y_test, pred))


              precision    recall  f1-score   support

           0       0.96      0.99      0.97     10961
           1       0.99      0.96      0.97     10961

    accuracy                           0.97     21922
   macro avg       0.97      0.97      0.97     21922
weighted avg       0.97      0.97      0.97     21922



In [7]:
## 모델 저장
import joblib

joblib.dump(vectorizer, "vectorizer.pkl")
joblib.dump(model, "phish_model.pkl")


['phish_model.pkl']

In [9]:
import os
os.listdir()

['.ipynb_checkpoints',
 'data',
 'frontend',
 'Phishing URL dataset.zip',
 'phish_model.pkl',
 'Phsing Data .ipynb',
 'Untitled.ipynb',
 'vectorizer.pkl']

In [14]:
import joblib

# 저장된 모델 불러오기
vectorizer = joblib.load("vectorizer.pkl")
model = joblib.load("phish_model.pkl")

# 테스트할 URL
test_url = "https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query=%EA%B0%95%EB%82%AD%ED%86%B5&ackey=xa213s6v"

# URL을 TF-IDF 벡터로 변환
X_test_single = vectorizer.transform([test_url])

# 예측
pred = model.predict(X_test_single)[0]

# 결과 해석
if pred == 1:
    print("🚨 피싱 URL입니다!")
else:
    print("✅ 정상 URL입니다!")


✅ 정상 URL입니다!
